# Bayesian Unfolding

This notebook contains the traditional **Bayesian unfolding** of the smeared data.

At first, we need to import ROOT with the RooUnfold library, and select data samples for training (creating the response matrix and unfolding matrix) and unfolding.

In [8]:
import ROOT
ROOT.gSystem.Load("libRooUnfold") # load RooUnfold
ROOT.gStyle.SetOptStat(0) # disables the stat boxes in plots

# select data for training and unfolding by their seed
seedTraining = 2
seedUnfolding = 22

### 1D Distributions

We load the histograms generated in the plot-ea.ipynb notebook.

In [9]:
# open files for reading
fileTraining = ROOT.TFile(f"data/{seedTraining}/events{seedTraining}_plots.root", "READ")
fileUnfolding = ROOT.TFile(f"data/{seedUnfolding}/events{seedUnfolding}_plots.root", "READ") # a different simulation -- represents the measured data

distributions = {'Nch': 'Multiplicity', 'S0pT1': 'Unweighted Spherocity'} # dictionary of distributions we want to unfold (e.g., multiplicity, spherocity, pT spectra, ...), syntax: abbreviation: full name
histos = {} # create an empty dictionary where we will store our histograms

# fill the distributions dictionary
for abbrev, name in distributions.items():
    # syntax: abbreviation: (name, true training data, smeared training data, response matrix from training data, independent true data, smeared data to be unfolded)
    histos[abbrev] = [fileTraining.Get(f"histo{seedTraining}{abbrev}{kind}") for kind in ["True", "Smeared", "RM"]] # fill in the training data
    histos[abbrev] += [fileUnfolding.Get(f"histo{seedUnfolding}{abbrev}{kind}") for kind in ["True", "Smeared"]] # append the data to be unfolded
    histos[abbrev].insert(0, name) # prepend the full name of the observable

Afterwards, we construct response matrices compatible with RooUnfold for each observable from the training data. Then, we proceed with the Bayesian unfolding of the measured data and plot all distributions for comparison. Lastly, we perform a closure test to evaluate the quality of unfolding.

In [10]:
# create a dictionary where unfolding matrices (UMs) will be stored
UMs = {}

# loop over different observables
for abbrev, data in histos.items():
    # define histogram variables from the histos dictionary
    name = data[0]
    histoTrainingTrue = data[1]
    histoTrainingSmeared = data[2]
    histoTrainingRM = data[3]
    histoUnfoldingTrue = data[4]
    histoUnfoldingSmeared = data[5]

    # ---------
    # UNFOLDING
    # ---------

    # create a RooUnfold response matrix
    response = ROOT.RooUnfoldResponse(histoTrainingSmeared, histoTrainingTrue, histoTrainingRM)

    # Bayesian unfolding with nIter iterations
    nIter = 5 # define the number of iterations
    unfolded = ROOT.RooUnfoldBayes(response, histoTrainingSmeared) # initialize the unfolding object
    
    unfolded.SetIterations(nIter - 1) # perform nIter-1 iterations
    histoPenultimate = unfolded.Hunfold().Clone("histoPenultimate") # extract the histogram after the penultimate iteration (we must use the .Clone() method otherwise defining histoUnfolded below would overwrite this histogram)
    histoPenultimate.SetDirectory(0) # makes sure that Python takes memory management ownership of the object

    unfolded.SetIterations(nIter) # perform nIter iterations
    histoUnfolded = unfolded.Hunfold().Clone("histoUnfolded") # extract the unfolded histogram (we must once again use the .Clone() method for several reasons: .Hunfold() returns a pointer to an internal histogram managed by the RooUnfold object; if we modified (e.g., scaled) the histogram without cloning it, it would alter the internal state of the unfolding engine and if we then wanted to extract some more information about the RooUnfold object (e.g., a covariance matrix), the math would be broken)
    histoUnfolded.SetDirectory(0)

    UMs[abbrev] = unfolded.UnfoldingMatrix() # extract the unfolding matrix (UM)

    # --------------
    # UNFOLDING PLOT
    # --------------

    # set up canvas for drawing
    canvasUnfolding = ROOT.TCanvas(f"canvasUnfolding{abbrev}", f"Bayesian Unfolding of {name}", 800, 600)
    canvasUnfolding.SetLogy() 

    # normalize the histograms
    for histo in [histoUnfoldingTrue, histoUnfoldingSmeared, histoPenultimate, histoUnfolded]:
        integral = histo.Integral()
        if integral > 0:
            histo.Scale(1.0 / integral)

    # customize the histograms (to customize title, labels, ..., we have to modify the first drawn histogram)
    histoUnfoldingTrue.SetTitle(f"Bayesian Unfolding of {name};Values;Normalized Events") # set title and labels of axes

    maxVal = max(histoUnfoldingTrue.GetMaximum(), histoUnfoldingSmeared.GetMaximum(), histoPenultimate.GetMaximum(), histoUnfolded.GetMaximum())
    histoUnfoldingTrue.SetMaximum(maxVal * 4.5) # scale the y-axis to prevent cut-off

    histoUnfoldingTrue.SetLineColor(ROOT.kGreen)
    histoUnfoldingSmeared.SetLineColor(ROOT.kBlue)
    histoPenultimate.SetLineColor(ROOT.kOrange)
    histoUnfolded.SetLineColor(ROOT.kRed)

    histoPenultimate.SetMarkerStyle(ROOT.kOpenSquare)
    histoPenultimate.SetMarkerColor(ROOT.kOrange)
    histoPenultimate.SetMarkerSize(0.8)

    histoUnfolded.SetMarkerStyle(ROOT.kFullCircle)
    histoUnfolded.SetMarkerColor(ROOT.kRed)
    histoUnfolded.SetMarkerSize(0.5)

    # drawing
    histoUnfoldingTrue.Draw("HIST")
    histoUnfoldingSmeared.Draw("HIST SAME") # "SAME" tells ROOT to draw the histogram on top of the previous one
    histoPenultimate.Draw("PE SAME") # "P" forces a point scatter
    histoUnfolded.Draw("PE SAME") # "E" ensures that error bars are drawn

    # legend
    legend = ROOT.TLegend(0.69, 0.74, 0.89, 0.89) # set coordinates as fractions of the canvas (x1, y1, x2, y2)
    legend.SetBorderSize(0) # removes the border around the legend box
    
    legend.AddEntry(histoUnfoldingTrue, "True Data", "l")
    legend.AddEntry(histoUnfoldingSmeared, "Smeared Data", "l")
    legend.AddEntry(histoPenultimate, f"{nIter - 1}. Iteration", "p")
    legend.AddEntry(histoUnfolded, f"{nIter}. Iteration", "p")
    
    legend.Draw()

    # save
    canvasUnfolding.SaveAs(f"img/{seedUnfolding}/{seedUnfolding}{abbrev}Unfolding.pdf") # save the canvas as a PDF file

    # ------------
    # CLOSURE TEST
    # ------------

    # define the histogram of the unfolded data and true data ratio
    histoRatio = histoUnfolded.Clone(f"closureRatio{abbrev}") # set the numerator = unfolded data of the ratio
    histoRatio.SetDirectory(0)
    histoRatio.Divide(histoUnfoldingTrue) # divide by the true data histogram

    # set up canvas
    canvasClosure = ROOT.TCanvas(f"canvasClosure{abbrev}", f"Closure Test for Unfolding of {name}", 800, 600)

    # customize the histogram
    histoRatio.SetTitle(f"Closure Test for Unfolding of {name};Values;Unfolded / True Data")

    histoRatio.SetMinimum(0.7)
    histoRatio.SetMaximum(1.3)

    histoRatio.SetLineColor(ROOT.kPink)
    histoRatio.SetMarkerStyle(ROOT.kFullCircle)
    histoRatio.SetMarkerColor(ROOT.kPink)
    histoRatio.SetMarkerSize(0.5)

    # draw the histogram
    histoRatio.Draw("PE") 
    
    # draw the reference line at y=1
    xMin = histoRatio.GetXaxis().GetXmin() # extract x-axis limits
    xMax = histoRatio.GetXaxis().GetXmax()
    line = ROOT.TLine(xMin, 1.0, xMax, 1.0) # create a line between xMin, xMax at y=1 (TLine(x1, y1, x2, y2))
    line.SetLineColor(ROOT.kGreen)
    line.SetLineStyle(2) # dashed line
    line.SetLineWidth(2)
    line.Draw("SAME")

    # save
    canvasClosure.SaveAs(f"img/{seedUnfolding}/{seedUnfolding}{abbrev}ClosureTest.pdf")

Using response matrix priors
Priors:

Vector (70)  is as follows

     |        1  |
------------------
   0 |0.000243503 
   1 |0.000854009 
   2 |0.00193052 
   3 |0.00347204 
   4 |0.00558056 
   5 |0.00829159 
   6 |0.0116136 
   7 |0.0153407 
   8 |0.0192262 
   9 |0.0236848 
  10 |0.0281288 
  11 |0.0322744 
  12 |0.0362824 
  13 |0.0396414 
  14 |0.0425695 
  15 |0.04494 
  16 |0.0465905 
  17 |0.047354 
  18 |0.047781 
  19 |0.0474005 
  20 |0.0461115 
  21 |0.0446655 
  22 |0.0427185 
  23 |0.0404399 
  24 |0.0377349 
  25 |0.0347969 
  26 |0.0320644 
  27 |0.0290093 
  28 |0.0262663 
  29 |0.0234523 
  30 |0.0206307 
  31 |0.0184077 
  32 |0.0156587 
  33 |0.0138162 
  34 |0.0119061 
  35 |0.0101521 
  36 |0.0086711 
  37 |0.00727508 
  38 |0.00609657 
  39 |0.00518456 
  40 |0.00421655 
  41 |0.00343654 
  42 |0.00285153 
  43 |0.00234353 
  44 |0.00184902 
  45 |0.00148552 
  46 |0.00121701 
  47 |0.00092851 
  48 |0.000736508 
  49 |0.000595507 
  50 |0.000461005 
  51 |0.

Info in <TCanvas::Print>: pdf file img/22/22NchUnfolding.pdf has been created
Info in <TCanvas::Print>: pdf file img/22/22NchClosureTest.pdf has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvasUnfoldingS0pT1
Info in <TCanvas::Print>: pdf file img/22/22S0pT1Unfolding.pdf has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvasClosureS0pT1
Info in <TCanvas::Print>: pdf file img/22/22S0pT1ClosureTest.pdf has been created


### $p_\mathrm{T}$ Spectra

We are going to need NumPy to perform operations to find quantiles later.

In [11]:
import numpy as np

In [ ]:
# load a 2D histograms of measured S0pT1 vs measured pT, true variant, and a S0pT1 unfolding matrix (UM)
histopTS0pT1True = fileUnfolding.Get(f"histo{seedUnfolding}pTS0pT1True")
histopTS0pT1Unfolding = fileUnfolding.Get(f"histo{seedUnfolding}pTS0pT1Smeared")
UMS0pT1 = UMs["S0pT1"]

# load the C++ applyUM(histoMeas, UM) function which applies the UM onto the y-axis
ROOT.gInterpreter.ProcessLine('#include "cpp/applyUM.cpp"')

# calculate the 2D histogram of unfolded S0pT1 vs measured pT
histopTS0pT1Unfolded = ROOT.applyUM(histopTS0pT1Unfolding, UMS0pT1)

# plot to visualize the 2D histogram of unfolded S0pT1 vs pT -- the labels and title is wrong, but its just for visualization
c1 = ROOT.TCanvas()
c1.SetLogz()
histopTS0pT1Unfolded.Draw("COLZ")
c1.Draw()

In file included from input_line_204:1:
./cpp/applyUM.cpp:7:7: error: redefinition of 'applyUM'
TH2D* applyUM(TH2D* histoMeas, TMatrixD* UM) {
      ^
./cpp/applyUM.cpp:7:7: note: previous definition is here
TH2D* applyUM(TH2D* histoMeas, TMatrixD* UM) {
      ^


Then, we obtain the $p_\mathrm{T}$ spectra for the top 1%, 5%, 10%, 90%, 95%, 99% $S_0^{p_\mathrm{T} = 1}$ events. 

Pozn. Gemini navrhuje, ze bych mel pro kazdou tridu sferocity udelat pT histogram z true a smeared data samples a response matici a vsechny unfoldovat (nebude to prilis vypocetne narocne?).

In [13]:
# project the 2D histograms onto the y-axis to get the distribution of the observable
histoS0pT1True = histopTS0pT1True.ProjectionY("histoS0pT1True")
histoS0pT1True.SetDirectory(0)

histoS0pT1Meas = histopTS0pT1Unfolding.ProjectionY("histoS0pT1Meas")
histoS0pT1Meas.SetDirectory(0)

histoS0pT1Unf = histopTS0pT1Unfolded.ProjectionY("histoS0pT1Unf")
histoS0pT1Unf.SetDirectory(0)

# define the quantile array with np.float64 data type to match ROOT's Double_t, which is required by the GetQuantiles() function
quantiles = np.array([0.01, 0.05, 0.1, 0.9, 0.95, 0.99], dtype=np.float64)

# define empty arrays for the output S0pT1
nQuantiles = len(quantiles)
valuesTrue = np.zeros(nQuantiles, dtype=np.float64)
valuesMeas = np.zeros(nQuantiles, dtype=np.float64)
valuesUnf = np.zeros(nQuantiles, dtype=np.float64)

# calculate the S0pT1 values for the given quantiles
histoS0pT1True.GetQuantiles(nQuantiles, valuesTrue, quantiles)
histoS0pT1Meas.GetQuantiles(nQuantiles, valuesMeas, quantiles)
histoS0pT1Unf.GetQuantiles(nQuantiles, valuesUnf, quantiles)

print(valuesTrue)
print(valuesMeas)
print(valuesUnf)

# get the pT spectra for each quantile
for i in range(nQuantiles):
    quantile = quantiles[i]

    # extract bin numbers for each quantile
    valueTrue = valuesTrue[i]
    valueMeas = valuesMeas[i]
    valueUnf = valuesUnf[i]

    binTrue = histopTS0pT1True.GetYaxis().FindBin(valueTrue)
    binMeas = histopTS0pT1Unfolding.GetYaxis().FindBin(valueMeas)
    binUnf = histopTS0pT1Unfolded.GetYaxis().FindBin(valueUnf)

    # calculate the x-axis projections 
    histopTTrue = histopTS0pT1True.ProjectionX(f"histo{seedUnfolding}pTTrueTop{quantile}S0pT1", binTrue, -1)
    histopTTrue.SetDirectory(0)

    histopTMeas = histopTS0pT1Unfolding.ProjectionX(f"histo{seedUnfolding}pTMeasTop{quantile}S0pT1", binMeas, -1)
    histopTMeas.SetDirectory(0)

    histopTUnf = histopTS0pT1Unfolded.ProjectionX(f"histo{seedUnfolding}pTUnfTop{quantile}S0pT1", binUnf, -1)
    histopTUnf.SetDirectory(0)

    # --------
    # PLOTTING
    # --------

    # set up canvas
    canvaspT = ROOT.TCanvas(f"canvaspTTop{quantile}S0pT1", "p_{T} spectrum for the top " + str(int(quantile * 100)) + "% S_{0}^{p_{T} = 1} events", 800, 600)
    canvaspT.SetLogy()

    # normalize the histograms
    for histo in [histopTTrue, histopTMeas, histopTUnf]:
        integral = histo.Integral()
        if integral > 0:
            histo.Scale(1.0 / integral)

    # customize the histograms
    histopTTrue.SetTitle("p_{T} spectrum for the top " + str(int(quantile * 100)) + "% S_{0}^{p_{T} = 1} events;p_{T};Normalized Events")

    histopTTrue.SetLineColor(ROOT.kGreen)
    histopTMeas.SetLineColor(ROOT.kBlue)
    histopTUnf.SetLineColor(ROOT.kRed)

    # drawing
    histopTTrue.Draw("HIST")
    histopTMeas.Draw("HIST SAME")
    histopTUnf.Draw("HIST SAME")

    # legend
    legend = ROOT.TLegend(0.69, 0.74, 0.89, 0.89)
    legend.SetBorderSize(0)
    
    legend.AddEntry(histopTTrue, "True Data", "l")
    legend.AddEntry(histopTMeas, "Smeared Data", "l")
    legend.AddEntry(histopTUnf, f"Unfolded Data", "l")
    
    legend.Draw()

    # save
    canvaspT.SaveAs(f"img/{seedUnfolding}/{seedUnfolding}pTTop{quantile}S0pT1.pdf")

    # ------------
    # CLOSURE TEST
    # ------------

    # define the histogram of the unfolded data and true data ratio
    histopTRatio = histopTUnf.Clone(f"closureRatiopTTop{quantile}S0pT1") # set the numerator = unfolded data of the ratio
    histopTRatio.SetDirectory(0)
    histopTRatio.Divide(histopTTrue) # divide by the true data histogram

    # set up canvas
    canvaspTClosure = ROOT.TCanvas(f"canvaspTClosureTop{quantile}S0pT1", "Closure Test for Unfolding of a p_{T} spectrum for the top " + str(int(quantile * 100)) + "% S_{0}^{p_{T} = 1} events", 800, 600)

    # customize the histogram
    histopTRatio.SetTitle("Closure Test for Unfolding of p_{T} spectrum for the top " + str(int(quantile * 100)) + "% S_{0}^{p_{T} = 1} events;Values;Unfolded / True Data")

    histopTRatio.SetMinimum(0.7)
    histopTRatio.SetMaximum(1.3)

    histopTRatio.SetLineColor(ROOT.kPink)
    histopTRatio.SetMarkerStyle(ROOT.kFullCircle)
    histopTRatio.SetMarkerColor(ROOT.kPink)
    histopTRatio.SetMarkerSize(0.5)

    # draw the histogram
    histopTRatio.Draw("PE") 
    
    # draw the reference line at y=1
    xMin = histopTRatio.GetXaxis().GetXmin() # extract x-axis limits
    xMax = histopTRatio.GetXaxis().GetXmax()
    line = ROOT.TLine(xMin, 1.0, xMax, 1.0) # create a line between xMin, xMax at y=1 (TLine(x1, y1, x2, y2))
    line.SetLineColor(ROOT.kGreen)
    line.SetLineStyle(2) # dashed line
    line.SetLineWidth(2)
    line.Draw("SAME")

    # save
    canvaspTClosure.SaveAs(f"img/{seedUnfolding}/{seedUnfolding}pTTop{quantile}S0pT1ClosureTest.pdf")

[0.16749059 0.25526331 0.31081371 0.77773387 0.82936197 0.89652916]
[0.11681956 0.20073765 0.25682579 0.75596881 0.80867248 0.87751485]
[0.1180913  0.20274242 0.25957962 0.76117702 0.81513102 0.88608085]


Info in <TCanvas::Print>: pdf file img/22/22pTTop0.01S0pT1.pdf has been created
Info in <TCanvas::Print>: pdf file img/22/22pTTop0.01S0pT1ClosureTest.pdf has been created
Info in <TCanvas::Print>: pdf file img/22/22pTTop0.05S0pT1.pdf has been created
Info in <TCanvas::Print>: pdf file img/22/22pTTop0.05S0pT1ClosureTest.pdf has been created
Info in <TCanvas::Print>: pdf file img/22/22pTTop0.1S0pT1.pdf has been created
Info in <TCanvas::Print>: pdf file img/22/22pTTop0.1S0pT1ClosureTest.pdf has been created
Info in <TCanvas::Print>: pdf file img/22/22pTTop0.9S0pT1.pdf has been created
Info in <TCanvas::Print>: pdf file img/22/22pTTop0.9S0pT1ClosureTest.pdf has been created
Info in <TCanvas::Print>: pdf file img/22/22pTTop0.95S0pT1.pdf has been created
Info in <TCanvas::Print>: pdf file img/22/22pTTop0.95S0pT1ClosureTest.pdf has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvaspTTop0.99S0pT1
Info in <TCanvas::Print>: pdf file img/22/22pTTop0.99S0pT1.pd

In the end, we close the files we opened in the beginning.

In [14]:
fileTraining.Close()
fileUnfolding.Close()